In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, f1_score

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [3]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))
class_weight_dict

{0: np.float64(1.1150922909880565),
 1: np.float64(1.6725583177659626),
 2: np.float64(0.6643075927934098)}

#### Logistic Regression
Instantiate the model

In [4]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight=class_weight_dict,
    random_state=42
)

Train

In [5]:
log_reg.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,"{0: np.float64(1.1150922909880565), 1: np.float64(1.6725583177659626), 2: np.float64(0.6643075927934098)}"
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


Predict

In [6]:
y_pred_lr = log_reg.predict(X_test)

Evaluate

In [7]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.63      0.67      0.65      2609
           1       0.77      0.92      0.84      1740
           2       0.88      0.78      0.83      4381

    accuracy                           0.77      8730
   macro avg       0.76      0.79      0.77      8730
weighted avg       0.78      0.77      0.78      8730



Logistic Regression achieved strong recall (0.92) for the Delayed class, indicating effective prioritization of high-risk orders. Performance on the At Risk class was comparatively lower due to overlap with adjacent categories, reflecting inherent label ambiguity. Macro-averaged F1-score (0.77) was used as the primary metric due to class imbalance.

What this tells you about Logistic Regression
Strengths

✅ Excellent recall for Delayed
✅ Interpretable baseline
✅ Properly handles imbalance with class weights

Limitations

❌ Struggles with At Risk (overlap class)
❌ Linear decision boundaries

This sets expectations for:

Trees

Random Forest

SVM

They should:

Improve At Risk separation

Possibly improve On-Time recall

Maintain Delayed recall

#### Decision Tree
Instantiate

In [8]:
dt_model = DecisionTreeClassifier(
    max_depth=10,
    class_weight=class_weight_dict,
    random_state=42
)

Train

In [9]:
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,"{0: np.float64(1.1150922909880565), 1: np.float64(1.6725583177659626), 2: np.float64(0.6643075927934098)}"


Predict

In [10]:
y_pred_dt = dt_model.predict(X_test)

Evaluate

In [11]:
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.70      0.76      0.73      2609
           1       0.79      0.91      0.85      1740
           2       0.92      0.82      0.87      4381

    accuracy                           0.82      8730
   macro avg       0.80      0.83      0.81      8730
weighted avg       0.83      0.82      0.82      8730



The Decision Tree classifier outperformed Logistic Regression across all classes, particularly improving recall and F1-score for the At Risk category. Performance on the Delayed class remained strong (recall = 0.91), while overall macro-averaged F1-score increased to 0.81, indicating better balanced multi-class performance.

Strengths

✅ Handles non-linear interactions
✅ Much better with the “At Risk” gray zone
✅ Interpretable (rules, splits)
✅ Responds well to class weights

Weaknesses (keep in mind for later)

⚠️ Can overfit if depth is not controlled
⚠️ Single tree = high variance

This naturally sets up Random Forest as the next step.

#### Random Forest
Instantiate

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

Train

In [13]:
rf_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Predict

In [14]:
y_pred_rf = rf_model.predict(X_test)

Evaluate

In [15]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.73      0.74      0.73      2609
           1       0.83      0.89      0.86      1740
           2       0.89      0.86      0.88      4381

    accuracy                           0.83      8730
   macro avg       0.82      0.83      0.82      8730
weighted avg       0.83      0.83      0.83      8730



The Random Forest classifier achieved the highest macro-averaged F1-score (0.82), indicating improved generalization over a single Decision Tree. While recall for the Delayed class slightly decreased compared to the tree model, precision improved, resulting in a more balanced overall performance across all classes.

What Random Forest actually bought you

✅ Better generalization
✅ Fewer extreme mistakes
✅ More stable predictions
✅ Best overall balance so far

But:

It didn’t dramatically change At Risk

It slightly softened Delayed recall

So… is Random Forest the winner?

So far, yes, if your goal is:

Best overall balance

Production-style robustness

But:

If the business says “Never miss delayed orders”,
Decision Tree might still be competitive due to slightly higher recall.

This tension is exactly what you want to discuss in evaluation.

In [16]:
results = {
    "Logistic Regression": f1_score(y_test, y_pred_lr, average="macro"),
    "Decision Tree": f1_score(y_test, y_pred_dt, average="macro"),
    "Random Forest": f1_score(y_test, y_pred_rf, average="macro")
}

results

{'Logistic Regression': 0.7706642066185275,
 'Decision Tree': 0.8149888332623808,
 'Random Forest': 0.823449551483917}

In [17]:
results_df = pd.DataFrame.from_dict(
    results, orient="index", columns=["Macro_F1"]
).sort_values(by="Macro_F1", ascending=False)

results_df

,Macro_F1
Random Forest,0.823450
Decision Tree,0.814989
Logistic Regression,0.770664


#### Cross validation
Logistic Regression

In [29]:
from sklearn.model_selection import cross_val_score

lr_cv_scores = cross_val_score(
    log_reg,
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro"
)

lr_cv_scores.mean(), lr_cv_scores.std()

(np.float64(0.7672526355027232), np.float64(0.0050362691855533135))

Decision Tree

In [27]:
dt_cv_scores = cross_val_score(
    dt_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro"
)
dt_cv_scores.mean(), dt_cv_scores.std()

(np.float64(0.8103214857985407), np.float64(0.0032420529678482742))

Random Forest

In [28]:
rf_cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro"
)
rf_cv_scores.mean(), rf_cv_scores.std()

(np.float64(0.8186474747905048), np.float64(0.004961886721920994))

In [30]:
cv_results = {
    "Logistic Regression": (lr_cv_scores.mean(), lr_cv_scores.std()),
    "Decision Tree": (dt_cv_scores.mean(), dt_cv_scores.std()),
    "Random Forest": (rf_cv_scores.mean(), rf_cv_scores.std())
}

cv_results

{'Logistic Regression': (np.float64(0.7672526355027232),
  np.float64(0.0050362691855533135)),
 'Decision Tree': (np.float64(0.8103214857985407),
  np.float64(0.0032420529678482742)),
 'Random Forest': (np.float64(0.8186474747905048),
  np.float64(0.004961886721920994))}

In [33]:
pd.DataFrame({
    "y_test": y_test,
    "y_pred_lr": y_pred_lr,
    "y_pred_dt": y_pred_dt,
    "y_pred_rf": y_pred_rf,
}).to_csv("../data/processed/model_predictions.csv", index=False)

In [34]:
import joblib

joblib.dump(rf_model, "../models/random_forest_model.pkl")

['../models/random_forest_model.pkl']